
# Overview
### Model training is an iterative and experimental process.

![image-alt-text](https://synapseaisolutionsa.blob.core.windows.net/public/imgs/flaml_modeladaptation.png)

In [ ]:
%pip install "flaml[synapse]@git+https://github.com/microsoft/FLAML.git@main"

In [ ]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

# Demo overview
|  | | | | |
|-----|-----|--------|--------|--------|
|![synapse](https://microsoft.github.io/SynapseML/img/logo.svg)| <img src="https://www.microsoft.com/en-us/research/uploads/prod/2020/02/flaml-1024x406.png" alt="drawing" width="200"/> | ![image-alt-text](https://th.bing.com/th/id/OIP.5aNnFabBKoYIYhoTrNc_CAHaHa?w=174&h=180&c=7&r=0&o=5&pid=1.7)| 


<style>
td, th {
   border: none!important;
}
</style>


#### 1. **Hyperparameter Tuning**: Given a dataset, model task (e.g. classification), model type (e.g. LightGBM), and range of parameters (e.g. number of iterations, number of leaves, max_depth), help me find the best parameters for a given metric (e.g. accuracy).
#### 2. **AutoML**: Given a dataset & model task, help me find the best model. 

![image-alt-text](https://synapseaisolutionsa.blob.core.windows.net/public/imgs/flaml_scenario.png)

## 2. Load data and preprocess

In [ ]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(
        "wasbs://publicwasb@mmlspark.blob.core.windows.net/company_bankruptcy_prediction_data.csv"
    )
)
# print dataset size
print("records read: " + str(df.count()))

In [ ]:
display(df)

### Featurize the dataset 

In [ ]:
train_raw, test_raw = df.randomSplit([0.8, 0.2], seed=41)
fit_raw, validation_raw = train_raw.randomSplit([0.8, 0.2], seed=42)

In [ ]:
from pyspark.ml.feature import VectorAssembler

feature_cols = df.columns[1:]
featurizer = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_data = featurizer.transform(train_raw)["Bankrupt?", "features"]
test_data = featurizer.transform(test_raw)["Bankrupt?", "features"]
fit_data = featurizer.transform(fit_raw)["Bankrupt?", "features"]
validation_data = featurizer.transform(validation_raw)["Bankrupt?", "features"]

## 3. Train the Default SynapseML Model

In [ ]:
import mlflow
mlflow.set_experiment("flaml_tune_demo")
mlflow.autolog(exclusive=False)

In [ ]:
def predict(model, test_data=validation_data):

    predictions = model.transform(test_data)
    
    metrics = ComputeModelStatistics(
        evaluationMetric="classification",
        labelCol="Bankrupt?",
        scoredLabelsCol="prediction",
    ).transform(predictions)
    return metrics


In [ ]:
from synapse.ml.lightgbm import LightGBMClassifier
from synapse.ml.train import ComputeModelStatistics

with mlflow.start_run(run_name="tune_default") as run:
    model = LightGBMClassifier(objective="binary", featuresCol="features", labelCol="Bankrupt?", isUnbalance=True)
    model = model.fit(train_data)

    # Generate predictions and log metrics
    default_metrics = predict(model)
    
    mlflow.log_metrics({"accuracy": default_metrics.first()['accuracy'], "AUC": default_metrics.first()['AUC']})

    default_metrics.show()

## 4. Tune the model with FLAML Tune

In [ ]:
def train(lambdaL1, learningRate, numLeaves, numIterations, train_data=fit_data, val_data=validation_data):
    """
    This train() function:
     - takes hyperparameters as inputs (for tuning later)
     - returns the AUC score on the validation dataset

    Wrapping code as a function makes it easier to reuse the code later for tuning.
    """
    lgc = LightGBMClassifier(
        objective="binary",
        lambdaL1=lambdaL1,
        learningRate=learningRate,
        numLeaves=numLeaves,
        labelCol="Bankrupt?",
        numIterations=numIterations,
        isUnbalance=True,
        featuresCol="features",
    )
    model = lgc.fit(train_data)
    # Define an evaluation metric and evaluate the model on the validation dataset.
    eval_metric = predict(model, val_data)
    eval_metric = eval_metric.toPandas()['AUC'][0]
    return model, eval_metric

In [ ]:
import flaml
import time

# define the search space
params = {
    "lambdaL1": flaml.tune.uniform(0.001, 1),
    "learningRate": flaml.tune.uniform(0.001, 1),
    "numLeaves": flaml.tune.randint(30, 100),
    "numIterations": flaml.tune.randint(100, 300),
}

# define the tune function
def flaml_tune(config):
    _, metric = train(**config)
    return {"auc": metric}

In [ ]:
with mlflow.start_run(nested=True, run_name="Child Run: "):
    analysis = flaml.tune.run(
        flaml_tune,
        params,
        time_budget_s=60,
        num_samples=100,
        metric="auc",
        mode="max",
        verbose=2,
        force_cancel=True,
        )

Best config and metric on validation data

In [ ]:
tune_config = analysis.best_config
tune_metrics_val = analysis.best_result
print("Best config: ", tune_config)
print("Best metrics on validation data: ", tune_metrics_val)

Retrain model on whole train_data and check metrics on test_data

In [ ]:
tune_model, tune_metrics = train(train_data=train_data, val_data=test_data, **tune_config)
print("Final held-out AUC:", tune_metrics)

## 4. Use AutoML to find the best model


In [ ]:
''' import AutoML class from the FLAML package '''
from flaml import AutoML
from flaml.automl.spark.utils import to_pandas_on_spark
automl = AutoML()

In [ ]:
import mlflow
mlflow.autolog()
mlflow.set_experiment("automl_classification_demo")

In [ ]:
import os
settings = {
    "time_budget": 120,  # total running time in seconds
    "metric": 'roc_auc',
    "task": 'classification',  # task type
    "log_file_name": 'flaml_experiment.log',  # flaml log file
    "seed": 42,    # random seed
    "force_cancel": True,  # force stop training once time_budget is used up
    "mlflow_exp_name": "automl_classification_demo"
}

In [ ]:
df = to_pandas_on_spark(train_data)
type(df)

In [ ]:
'''The main flaml automl API'''

with mlflow.start_run(nested=True):
    automl.fit(dataframe=df, label='Bankrupt?', isUnbalance=True, **settings)

### Best model and metric

In [ ]:
''' retrieve best config'''
print('Best hyperparmeter config:', automl.best_config)
print('Best roc_auc on validation data: {0:.4g}'.format(1-automl.best_loss))
print('Training duration of best run: {0:.4g} s'.format(automl.best_config_train_time))

In [ ]:
automl_metrics = predict(automl.model.estimator)
automl_metrics.show()

## 5. Use Apache Spark to Parallelize AutoML trials

In [ ]:
import mlflow
mlflow.autolog()
mlflow.set_experiment("automl_spark_demo")

In [ ]:
settings = {
    "time_budget": 120,  # total running time in seconds
    "metric": 'roc_auc',  # primary metrics for regression can be chosen from: ['mae','mse','r2','rmse','mape']
    "task": 'classification',  # task type    
    "seed": 7654321,    # random seed
    "use_spark": True,
    "n_concurrent_trials": 3,
    "force_cancel": True,
    "mlflow_exp_name": "automl_spark_demo"

}

In [ ]:
pandas_df = train_raw.toPandas()
pandas_df.head()

In [ ]:
'''The main flaml automl API'''
with mlflow.start_run(nested=True):
    automl.fit(dataframe=pandas_df, label='Bankrupt?', **settings)

In [ ]:
''' retrieve best config'''
print('Best hyperparmeter config:', automl.best_config)
print('Best roc_auc on validation data: {0:.4g}'.format(1-automl.best_loss))
print('Training duration of best run: {0:.4g} s'.format(automl.best_config_train_time))